In [1]:
import json 
import numpy as np
import pandas as pd

In [ ]:
data = ".././data/processed/F_tr1_d1.json"

with open(data, "r", encoding="UTF-8") as fp:
    data = json.load(fp)
data['gnss_data'].keys()

dict_keys(['lat', 'lon', 'height_ellipsoid', 'height_msl', 'ground_speed', 'vel_east', 'vel_north', 'vel_down', 'speed_acc', 'horizontal_acc', 'vertical_acc', 'heading_acc', 'time_acc', 'timestamp', 'hour', 'min', 'sec', 'nano', 'itow', 'nb_sats', 'position_dop', 'heading', 'fix_type', 'flags', 'flags2', 'utm32n_easting', 'utm32n_northing'])

In [3]:
data['gnss_data']['lat'][0], data['gnss_data']['lon'][0]

(44.950901, 6.8884617)

In [2]:
def transform(x):
    x = x.strip().replace("°", "").replace("'", "").split()[:-1]
    return float(x[0]) + float(x[1])/60 + float(x[2])/3600

In [4]:
porte = pd.read_csv(".././data/pointsLocationSecondCourse.csv", sep=";", header=0, index_col=0)
porte['WGS84_Lat2']  = porte['WGS84 Latitudine [°]'].apply(transform)
porte['WGS84_Lon2']  = porte['WGS84 Longitudine [°]'].apply(transform)
porte.to_csv(".././data/pointsLocationSecondCourse.csv", sep=";", index=True)

In [6]:
import folium

def plot_map_with_gates(data: dict, porte_df, save_map: str, filename: str) -> None:
    folium_map = folium.Map(
        location=[data['gnss_data']['lat'][0], data['gnss_data']['lon'][0]],
        zoom_start=15
    )

    # Skier trajectory points
    for lat, lon in zip(data['gnss_data']['lat'], data['gnss_data']['lon']):
        folium.CircleMarker(location=[lat, lon], radius=2, color='blue', fill=True, fill_opacity=0.8).add_to(folium_map)

    # Gate positions (WGS84_Lat2/WGS84_Lon2)
    for lat, lon in zip(porte_df['WGS84_Lat2'], porte_df['WGS84_Lon2']):
        folium.CircleMarker(location=[lat, lon], radius=4, color='red', fill=True, fill_opacity=0.9).add_to(folium_map)

    folium_map.save(save_map + f"/{filename}.html")

plot_map_with_gates(data, porte, ".././data/results", "trajectory_with_gates")